# OSU DEMO

In [1]:
# This file expects to be in juptyer/courses/fault101. If its not working, either move using cd inside the notebook, or adjust file locations of other commands.
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_NEORV32'
SS_VER = 'SS_VER_2_1'

In [3]:
pwd

'/Users/patrickgould/Projects/ChipWhisperer/chipwhisperer/jupyter/courses/fault101'

In [4]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍


In [5]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch-custom
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j OPT=0

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
riscv64-unknown-elf-gcc (g04696df09) 14.2.0
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CW308_NEORV32 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
-en     .././hal//neorv32/neorv32_gpio.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
-en     .././hal//neorv32/neorv32_cpu.c ...
Compiling:
-en     .././hal//neorv32/neorv32_cfs.c ...
-en     .././hal//neorv32/neorv32_mtime.c ...
Compiling:
Compiling:
.
.
.
.
.
.
-en     simpleserial-glitch-custom.c ...
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
.
Compiling:
.
.
.
-en     .././hal//neorv32/neorv32_gptmr.c ...
.
-en     .././hal//neorv32/neorv32_trng.c ...
Compiling:
Compiling:
Compiling:
Compil

.././hal//neorv32/syscalls.c:104:6: warning: "/*" within comment [-Wcomment]
  104 |     //*(volatile int *)EXIT_REG = exit_status;
.././hal//neorv32/syscalls.c:135:19: warning: 'struct timeb' declared inside parameter list will not be visible outside of this definition or declaration
  135 | int _ftime(struct timeb *tp)
      |                   ^~~~~
.././hal//neorv32/syscalls.c: In function '_sbrk':
.././hal//neorv32/syscalls.c:267:22: warning: comparison between two arrays [-Warray-compare]
  267 |     if (__heap_start == __heap_end) {
      |                      ^~
.././hal//neorv32/syscalls.c:267:22: note: use '&__heap_start[0] == &__heap_end[0]' to compare the addresses


-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     simpleserial-glitch-custom-CW308_NEORV32.elf ...
Memory region         Used Size  Region Size  %age Used
             ram:        5632 B        64 KB      8.59%
             rom:        9876 B        64 KB     15.07%
           iodev:           0 B        512 B      0.00%
-e Done!
.
.
.
.
.
Creating load file for Flash: simpleserial-glitch-custom-CW308_NEORV32.hex
riscv64-unknown-elf-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature simpleserial-glitch-custom-CW308_NEORV32.elf simpleserial-glitch-custom-CW308_NEORV32.hex
Creating Extended Listing: simpleserial-glitch-custom-CW308_NEORV32.lss
Creating Symbol Table: simpleserial-glitch-custom-CW308_NEORV32.sym
riscv64-unknown-elf-objdump -h -S -z simpleserial-glitch-custom-CW308_NEORV32.elf > simpleserial-glitch-custom-CW308

In [6]:
# Flash program, init comms.
fw_path = "../../../firmware/mcu/simpleserial-glitch-custom/simpleserial-glitch-custom-{}.bin".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [7]:
# Define reboot function. This is different for the ICE40.
def reboot_flush():
    #reset_target(scope) # <-- use this for other boards!
    # no reboot on the ICE40 since it doesn't have a way to reset it externally. We just need to reprogram it.
    cw.program_target(scope, prog, fw_path) 
    #Flush garbage too
    target.flush()

In [8]:
# Set default glitch parameters.
scope.cglitch_setup()

In [9]:
# Define define graph parameters
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [10]:
# Define graph.
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="width", y_index="offset")

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,offset]
      .Points.II :Points   [width,offset]

In [29]:
from tqdm.notebook import tqdm
import re
import struct
import time

sample_size = 1

# Range of error for target instruction, in cycles.
ERROR_TOLERANCE = 25
# Cycle instruction was executed in RTL simulation.
TARGET_CYCLE = 508

program_max_cycle_count = 1350 # Number of cycles a normal execution takes.
#gc.set_range("width", 3500, 4500)
gc.set_range("width", 4000, 4500)
#gc.set_range("offset", 1300, 3200)
gc.set_range("offset", 2100, 2800)
gc.set_global_step([400, 200, 100])
gc.set_range("ext_offset", 200, TARGET_CYCLE + ERROR_TOLERANCE)
gc.set_step("ext_offset", 1) # We are interested in the whole cycle search space; set step to 1.

scope.glitch.repeat = 1
reboot_flush()
broken = False
scope.adc.timeout = 0.5
hitList = list()        # Holds time and parameters for successful runs
failList = list()       # Holds time and parameters for crashed runs
normalList = list()     # Holds time and parameters for benign runs

clock_ID = time.CLOCK_MONOTONIC # Clock considers time since boot, including time the system has been suspended.
start_time = time.clock_gettime(clock_ID) # Get some time, in seconds. This will be our start time.
end_time = start_time # We will update this value each iteration.

for glitch_settings in gc.glitch_values():
    #start_time = time.time_ns()
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    for i in range(sample_size):
        if scope.adc.state:
            # can detect crash here (fast) before timing out (slow)
            failList.append((scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset))
            gc.add("reset")
            #Device is slow to boot?
            reboot_flush()

        scope.arm()
        target.simpleserial_write('p', bytearray([0]*5))
        ret = scope.capture()

        # Gather list elements; number of cycles + parameters. ADC counts 4 times each cycle, so we need to divide its count by 4.
        listElement = (scope.adc.trig_count/4, scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
        
        if ret:
            failList.append(listElement)
            gc.add("reset")
            
            #Device is slow to boot?
            reboot_flush()
        else:
            val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50) #For loop check
            if val['valid'] is False:
                failList.append(listElement)
                gc.add("reset")
            else:

                if val['payload'] == bytearray([1]): #for loop check
                    hitList.append(listElement)
                    gc.add("success")
                else:
                    normalList.append(listElement)
                    gc.add("normal")
                    
        end_time = time.clock_gettime(clock_ID) # Update timer

(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x52, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:418) Unexpected length 84, 1
(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x52, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:418) Unexpected length 84, 1
(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x65, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:425) Invalid CRC. Expected 113 got 235
(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x52, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:418) Unexpected length 84, 1
(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Unexpected start to command 0x52, expected 0x72
(ChipWhisperer Target WARNING|File SimpleSerial2.py:418) Unexpected length 84, 1
(ChipWhisperer Target WARNING|File SimpleSerial2.py:385) Une

USBErrorNoDevice: LIBUSB_ERROR_NO_DEVICE [-4]

In [ ]:
end_time-start_time

In [27]:
hitList

[]

In [58]:
failList

[(28843.0, 4300, 2800, 202),
 (26906.0, 4300, 2800, 203),
 (26906.0, 4300, 2800, 204),
 (28843.0, 4300, 2800, 207),
 (26389.0, 4300, 2800, 208),
 (26389.0, 4300, 2800, 209),
 (28842.0, 4300, 2800, 212),
 (26968.0, 4300, 2800, 215),
 (26968.0, 4300, 2800, 216),
 (28843.0, 4300, 2800, 219),
 (26388.0, 4300, 2800, 220),
 (28661.0, 4300, 2800, 229),
 (26575.0, 4300, 2800, 233),
 (37969.25, 4300, 2800, 234),
 (28843.0, 4300, 2800, 238),
 (38027.0, 4300, 2800, 240),
 (26575.0, 4300, 2800, 241),
 (26575.0, 4300, 2800, 243),
 (26575.0, 4300, 2800, 244),
 (28843.0, 4300, 2800, 247),
 (37324.0, 4300, 2800, 248),
 (37960.25, 4300, 2800, 249),
 (28843.0, 4300, 2800, 252),
 (22662.0, 4300, 2800, 254),
 (26574.0, 4300, 2800, 255),
 (26574.0, 4300, 2800, 257),
 (28841.0, 4300, 2800, 261),
 (28843.0, 4300, 2800, 264),
 (26906.0, 4300, 2800, 265),
 (28843.0, 4300, 2800, 269),
 (26451.0, 4300, 2800, 271),
 (28842.0, 4300, 2800, 274),
 (26968.0, 4300, 2800, 277),
 (26968.0, 4300, 2800, 278),
 (28843.0, 4

In [57]:
# Dump results into .txt files
with open('failList.txt', 'w') as file:
    for item in failList:
        file.write(str(item) + "," + '\n')
with open('hitList.txt', 'w') as file:
    for item in hitList:
        file.write(str(item) + "," + '\n')
with open('normalList.txt', 'w') as file:
    for item in normalList:
        file.write(str(item) + "," + '\n')

In [37]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

[((523,),
  {'total': 20,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.45,
   'normal': 11,
   'normal_rate': 0.55}),
 ((522,),
  {'total': 11,
   'success': 0,
   'success_rate': 0.0,
   'reset': 8,
   'reset_rate': 0.7272727272727273,
   'normal': 3,
   'normal_rate': 0.2727272727272727}),
 ((521,),
  {'total': 14,
   'success': 0,
   'success_rate': 0.0,
   'reset': 8,
   'reset_rate': 0.5714285714285714,
   'normal': 6,
   'normal_rate': 0.42857142857142855}),
 ((520,),
  {'total': 21,
   'success': 0,
   'success_rate': 0.0,
   'reset': 15,
   'reset_rate': 0.7142857142857143,
   'normal': 6,
   'normal_rate': 0.2857142857142857}),
 ((519,),
  {'total': 19,
   'success': 0,
   'success_rate': 0.0,
   'reset': 16,
   'reset_rate': 0.8421052631578947,
   'normal': 3,
   'normal_rate': 0.15789473684210525}),
 ((518,),
  {'total': 15,
   'success': 0,
   'success_rate': 0.0,
   'reset': 4,
   'reset_rate': 0.26666666666666666,
   'normal': 11,
   'normal_

And one for your width/offset settings:

In [52]:
results = gc.calc(sort="total")
results

[((4300, 1700, 525),
  {'total': 18,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.5,
   'normal': 9,
   'normal_rate': 0.5}),
 ((4300, 1700, 520),
  {'total': 18,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.5,
   'normal': 9,
   'normal_rate': 0.5}),
 ((3900, 2500, 525),
  {'total': 18,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.5,
   'normal': 9,
   'normal_rate': 0.5}),
 ((3900, 2500, 520),
  {'total': 18,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.5,
   'normal': 9,
   'normal_rate': 0.5}),
 ((3900, 2500, 514),
  {'total': 18,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.5,
   'normal': 9,
   'normal_rate': 0.5}),
 ((3900, 2500, 513),
  {'total': 18,
   'success': 0,
   'success_rate': 0.0,
   'reset': 9,
   'reset_rate': 0.5,
   'normal': 9,
   'normal_rate': 0.5}),
 ((3900, 2500, 512),
  {'total': 18,
   'success': 0,
   'succes

In [39]:
scope.dis()
target.dis()

In [ ]:
assert broken is True